## For quick code and convention tests

### 1. Kronecker (tensor) product

In [1]:
import numpy as np

I2 = np.eye(2, dtype=np.complex128)
Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=np.complex128)
Z1 = np.kron(Z, I2)
Z2 = np.kron(I2, Z)

print(Z1)
print(Z2)

[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j -1.+0.j -0.+0.j]
 [ 0.+0.j  0.+0.j -0.+0.j -1.+0.j]]
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j  0.+0.j -0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j -0.+0.j  0.+0.j -1.+0.j]]


So the matrix correspond to basis ordering |00>, |01>, |10>, |11> as usual.

### 2. Lindblad dynamics (QuGradLab --> QuGrad --> PySTE logic)

We will ignore how PySTE works and focuse on what QuGradLab and QuGrad need. 

QuGrad creates instances of QuantumSystem. To propagate them use sys.propagate(drive_amps, drive_frequancies, ctrl_amps, initial_state, dt), or I call it sys.propagate(*parameters). QuGradLab creates instances of Spinchain, which can be propagated similarly.

PulseForm unwraps an array x into the *parameters required, so that whenever you need to supply *parameters as inputs, you just supply x. This allows e.g. optimisation of x with scipy.minimize. 

When you transform something (QuantumSystem or SpinChain) into a LindbladSystem, the Hamiltonians are transformed automatically and all the methods work exactly as before. The only difference is that the *parameters you supply should contain initial state as a density matrix (before it is state vector). So similarly if you are using PulseForm to wrap *parameters into x, make sure the output is a density matrix.

Below is an example with zero control amps. (x = -1 because I think in PySTE convention, x = -1 correspond to Jmin and +1 corresponds to Jmax which you set manually at the start).

In [ ]:
import numpy as np
import tensorflow as tf
from qugradlab.systems.semiconducting.esr import SpinChain

# Define SpinChain with QuGradLab

sys = SpinChain(
    spins=2,
    zeeman_splittings=np.array([0.0, 0.0]),
    max_drive_strength=0.0,
    J_max=10.0,  # MHz
    J_min=0.0,
    feromagnetic=False,
    use_graph=True,
)

# Transform it to LindbladSystem with a depolarising channel (but lindblad operators set to zero) so we expect unitary evolution

I2 = np.eye(2, dtype=np.complex128)
X = np.array([[0, 1], [1, 0]], dtype=np.complex128)
Y = np.array([[0, -1j], [1j, 0]], dtype=np.complex128)
Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=np.complex128)

def local_depolarising_ops_2q(gamma):
    pref = np.sqrt(gamma / 4.0)

    return np.array([
        pref * np.kron(X, I2),
        pref * np.kron(Y, I2),
        pref * np.kron(Z, I2),
        pref * np.kron(I2, X),
        pref * np.kron(I2, Y),
        pref * np.kron(I2, Z),
    ], dtype=np.complex128)

L0 = local_depolarising_ops_2q(gamma=0.01)
sys_noisy = sys.add_lindblad_terms(L0=L0)

In [13]:
# Define the pulse function wrapper and transform into PulseForm. For my case I want to optimise the J coupling amplitudes so x will be J amplitudes (scaled between -1 and 1).

n_steps = 10
dt = 100.0 * 1e-3  # timestep in μs to match J in MHz
psi0 = np.array([0.0, 1.0, 0.0, 0.0], dtype=np.complex128)  # |01> state

def convert_dtype(np_array) -> tf.Tensor: 
    return tf.cast(tf.convert_to_tensor(np_array), dtype=tf.complex128)

def pulse_function(x):
    drive_amps = convert_dtype(np.zeros((n_steps, 2), dtype=np.complex128))      # drive amplitudes (disabled, 2 because there are 2 qubits so 2 channels)
    drive_frequencies = convert_dtype(np.array([0.0, 0.0], dtype=np.complex128)) # drive frequencies (disabled)
    x = convert_dtype(tf.reshape(x, (n_steps, 1)))                               # 1 because only 1 J channel for 2 qubits
    initial_state = convert_dtype(np.outer(psi0, psi0.conj()))                   # density matrix for |01><01|
    return (
        drive_amps,      
        drive_frequencies,   
        x,                                 
        initial_state,      
        dt * 2 * np.pi,          
    )

driven_sys_noisy = sys_noisy.pulse_form(pulse_function)

In [ ]:
# Run. A little bit of depolarisation as expected.

x = np.array([-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0], dtype=np.complex128)

final_state_noisy = driven_sys_noisy.propagate(x)
print(np.round(final_state_noisy, 3))

[[ 0.03 -0.j  0.   -0.j -0.   +0.j -0.   -0.j]
 [ 0.   +0.j  0.94 -0.j  0.   +0.j -0.   -0.j]
 [-0.   -0.j  0.   -0.j  0.001+0.j  0.   -0.j]
 [-0.   +0.j  0.   -0.j  0.   -0.j  0.03 +0.j]]


### 3. Suzuki-Trotter discretisation test

In [ ]:
import numpy as np
import tensorflow as tf
from qugradlab.systems.semiconducting.esr import SpinChain

psi0 = np.array([0.0, 1.0, 0.0, 0.0], dtype=np.complex128)  # |01> state
n_steps = 50
x_random = np.random.uniform(-1.0, 1.0, size=(n_steps, 1))  # random J amplitudes for testing

# Define SpinChain with QuGradLab

sys = SpinChain(
    spins=2,
    zeeman_splittings=np.array([0.0, 0.0]),
    max_drive_strength=0.0,
    J_max=10.0,  # MHz
    J_min=0.0,
    feromagnetic=False,
    use_graph=True,
)

def pulse_function(x):
    drive_amps = convert_dtype(np.zeros((n_steps, 2), dtype=np.complex128))      # drive amplitudes (disabled, 2 because there are 2 qubits so 2 channels)
    drive_frequencies = convert_dtype(np.array([0.0, 0.0], dtype=np.complex128)) # drive frequencies (disabled)
    x = convert_dtype(tf.reshape(x, (n_steps, 1)))                               # 1 because only 1 J channel for 2 qubits
    initial_state = convert_dtype(psi0)                   # density matrix for |01><01|
    return (
        drive_amps,      
        drive_frequencies,   
        x,                                 
        initial_state,      
        dt * 2 * np.pi,          
    )

driven_sys = sys.pulse_form(pulse_function)

